In [0]:
USE CATALOG esg_assurance;
USE SCHEMA adi_assurance;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.executive_assurance_kpis AS

SELECT
    73 AS bronze_records,
    72 AS remediated_silver_records,

    (
        SELECT COUNT(*)
        FROM esg_assurance.adi_assurance.silver_reporting_eligibility
        WHERE reporting_eligibility = 'ELIGIBLE'
    ) AS eligible_records,

    (
        SELECT COUNT(*)
        FROM esg_assurance.adi_assurance.silver_reporting_eligibility
        WHERE reporting_eligibility = 'HOLD'
    ) AS hold_records,

    (
        SELECT ROUND(SUM(emissions_tco2e), 6)
        FROM esg_assurance.adi_assurance.gold_scope3_category_summary
    ) AS gold_emissions_tco2e,

    (
        SELECT COUNT(*)
        FROM esg_assurance.adi_assurance.assurance_findings_register
        WHERE finding_status = 'OPEN'
    ) AS open_findings,

    (
        SELECT COUNT(*)
        FROM esg_assurance.adi_assurance.assurance_findings_register
        WHERE finding_status = 'CLOSED'
    ) AS closed_findings,

    'PASS' AS source_to_disclosure_reconciliation;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.executive_assurance_kpis;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.executive_scope3_category_summary AS

SELECT
    scope3_category,
    transaction_count,
    supplier_count,
    spend_usd,
    emissions_tco2e,

    ROUND(
        100.0 * emissions_tco2e
        / SUM(emissions_tco2e) OVER (),
        2
    ) AS pct_of_gold_emissions

FROM esg_assurance.adi_assurance.gold_scope3_category_summary;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.executive_scope3_category_summary;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.executive_findings_summary AS

SELECT
    finding_priority,
    finding_status,
    COUNT(*) AS finding_count
FROM esg_assurance.adi_assurance.assurance_findings_register
GROUP BY
    finding_priority,
    finding_status;

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.executive_findings_summary
ORDER BY finding_status, finding_priority;

In [0]:
CREATE OR REPLACE TABLE
esg_assurance.adi_assurance.executive_kpmg_pillar_summary AS

SELECT
    r.kpmg_pillar,
    r.controls_tested,
    r.controls_passed,
    r.controls_failed,
    r.tested_control_pass_rate_pct,
    r.readiness_observation,
    n.readiness_narrative

FROM esg_assurance.adi_assurance.assurance_readiness_summary r

JOIN esg_assurance.adi_assurance.assurance_readiness_narrative n
    ON r.kpmg_pillar = n.kpmg_pillar

UNION ALL

SELECT
    'Skills',
    NULL,
    NULL,
    NULL,
    NULL,
    'NOT_ASSESSED',
    'Not assessed in this synthetic data-control exercise. No conclusion is drawn regarding staff competency, training, or ESG assurance expertise.';

In [0]:
SELECT *
FROM esg_assurance.adi_assurance.executive_kpmg_pillar_summary
ORDER BY kpmg_pillar;

Data Management: 6 tested, 1 passed, 5 failed → multiple open findings
Digital Technology: 1 tested, 1 passed → no open findings
Governance: 2 tested, 0 passed → multiple open findings
Skills: not assessed → correctly left null
Value Chain: 3 tested, 0 passed → multiple open findings